# executor

> A persistent-namespace python runner with an approval gate: rishi's `run_py` semantics
> (stdout + last-expression repr) but no import/socket sandbox, so network skills like fossick work.

In [ ]:
#| default_exp executor

In [ ]:
#| hide
from nbdev.showdoc import *

rishi's built-in `run_py` executes through `safepyrun`, which blocks `socket` and `importlib` --
correct for untrusted code, fatal for skills whose whole point is the network (fossick) or dynamic
import (loading pyskills). ramabana replaces the syscall-level sandbox with an *approval gate*: every
snippet is offered to `approve` before it runs, in the same `{'function': {'name': ..., 'arguments':
...}}` shape rishi's tool loop uses, so `rishi.core.hitl_policy({'run_py': 'check'})` works unchanged.

The namespace persists across calls -- and across conversation compaction, since it lives here and
not in the model's context.

In [ ]:
#| export
import io, ast
from contextlib import redirect_stdout

In [ ]:
#| export
def code_call(code):
    'Wrap a snippet in the tool-call dict shape rishi\'s `approve`/`hitl_policy` expect.'
    return {'function': {'name': 'run_py', 'arguments': {'code': code}}}

In [ ]:
#| export
class Executor:
    'Run python snippets in one persistent namespace, gated by `approve`, returning stdout + last-expr repr.'
    def __init__(self,
                 ns:dict=None,      # namespace to run in (created if None); persists across calls
                 approve=None,      # approve(tool_call)->bool, rishi-compatible; None = allow all
                 max_len:int=4000): # truncate returned output beyond this many chars
        self.ns = ns if ns is not None else {}
        self.ns.setdefault('__name__', '__ramabana__')
        self.approve,self.max_len = approve,max_len
    def __call__(self, code:str) -> str:
        if self.approve is not None and not self.approve(code_call(code)): return 'Denied by human operator'
        buf = io.StringIO()
        try:
            tree = ast.parse(code)
            last = tree.body[-1] if tree.body and isinstance(tree.body[-1], ast.Expr) else None
            body = tree.body[:-1] if last is not None else tree.body
            with redirect_stdout(buf):
                if body: exec(compile(ast.Module(body=body, type_ignores=[]), '<ramabana>', 'exec'), self.ns)
                res = eval(compile(ast.Expression(body=last.value), '<ramabana>', 'eval'), self.ns) if last is not None else None
            out = buf.getvalue()
            if res is not None: out = (out.rstrip('\n') + '\n' if out else '') + repr(res)
            out = out or '(ok)'
        except Exception as e:
            pre = buf.getvalue()
            out = (pre + '\n' if pre else '') + f'{type(e).__name__}: {e}'
        if len(out) > self.max_len: out = out[:self.max_len] + f'\n... [truncated {len(out)-self.max_len} chars]'
        return out

In [ ]:
ex = Executor()
assert ex('x = 2') == '(ok)'
assert ex('x + 1') == '3'                          # namespace persists, last expr repr'd
assert ex('print("hi"); x*2') == 'hi\n4'           # stdout + repr
assert ex('1/0').startswith('ZeroDivisionError')   # exceptions come back as text, loop continues
assert ex('import json; json.dumps({"a": 1})') == '\'{"a": 1}\''

In [ ]:
# approval gate: rishi-style policy dicts work as-is
deny = Executor(approve=lambda tc: tc['function']['name'] != 'run_py')
assert deny('x = 1') == 'Denied by human operator' and 'x' not in deny.ns
seen = []
audit = Executor(approve=lambda tc: seen.append(tc['function']['arguments']['code']) or True)
assert audit('7*6') == '42' and seen == ['7*6']

In [ ]:
long = Executor(max_len=10)
assert long('"a"*50').startswith("'aaaaaaaaa") and 'truncated' in long('"a"*50')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()